# Barrido de topologías de arreglo — **análisis**

Notebook **exclusivamente de análisis**: no corre simulaciones, no genera RIRs y no
toca el dataset. Lee los resultados ya calculados por
[`array_topology_sweep_full.ipynb`](array_topology_sweep_full.ipynb)
(`tests/array/dataset_out_full/ism_benchmark_metrics.parquet`) y los interpreta.

**Pregunta:** con la misma **superficie ocupada** y el mismo número de micrófonos,
¿qué **topología planar** (circular, grilla, espiral, concéntrica) es más robusta ante
posiciones arbitrarias de locutor e interferencia dentro de una sala?

---

### Índice

**Parte 1 — Condiciones de la simulación** *(qué se corrió)*
1. Diseño factorial y cobertura del dataset
2. Geometrías del arreglo — misma superficie, distinto reparto
3. Escena 3D: sala, bafle, toroide de locutores y nube de interferencias

**Parte 2 — Resultados** *(qué salió)*
Cuatro métricas, todas como **Δ respecto del micrófono de referencia crudo**
(`Delta_tot_*_early`, agregadas por **mediana**):

| Métrica | Qué mide | Unidad |
|---|---|---|
| **PESQ** | calidad perceptual de la voz | Δ MOS-LQO |
| **STOI** | inteligibilidad | Δ (0–1) |
| **SI-SDR** | fidelidad de forma de onda, invariante a escala | Δ dB |
| **SIR** | rechazo específico de la interferencia | Δ dB |

> **Por qué mediana y no media:** la distribución de SIR tiene cola pesada (escenas con
> RT bajo y buena separación angular dan valores enormes). La media las deja mandar el
> ranking; la mediana no. Ver [[bss-eval-median-aggregation]].

## Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

REPO_ROOT = "/home/matias/Documents/Tesis/Vision-Aided-Beamformer"
SRC       = os.path.join(REPO_ROOT, "src")
DATA_DIR  = os.path.join(REPO_ROOT, "tests/array/dataset_out_full")
for p in (REPO_ROOT, SRC):
    if p not in sys.path:
        sys.path.insert(0, p)

# Unico import del motor: reconstruir la GEOMETRIA exacta que se simulo.
from beamforming.array.geometry import (generate_array_coords, place_spherical,
                                        select_reference_mic)

_pq, _csv = (os.path.join(DATA_DIR, f'ism_benchmark_metrics.{e}') for e in ('parquet', 'csv'))
try:
    df, _src = pd.read_parquet(_pq), _pq
except Exception:
    df, _src = pd.read_csv(_csv), _csv

print(f'[*] dataset : {os.path.relpath(_src, REPO_ROOT)}')
print(f'[*] filas    : {len(df):,}  x  {df.shape[1]} columnas')
print(f'[*] costo    : {df["exec_time_s"].sum()/3600:.2f} h de computo acumulado')

### Sistema visual

Una paleta por **rol semántico**, no por orden de aparición:

- **Topologías** → [Okabe–Ito](https://jfly.uni-koeln.de/color/): cualitativa, segura
  para dicromatismo y distinguible en escala de grises (impresión de la tesis).
  Cada topología lleva además **marcador propio**, para no depender solo del color.
- **M (micrófonos)** → rampa **secuencial fría**: más micros ⇒ más oscuro.
- **RT60** → rampa **secuencial cálida**: más reverberación ⇒ más oscuro.
- **Procesadores** → el sistema real (`NM-MVDR`) va en tinta llena; el `Souden oracle`,
  que es el **techo alcanzable** y no un competidor, va en gris.

In [ ]:
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

# ------------------------------- PALETA -------------------------------
INK, MUTED, GRIDC, EDGE, PAPER = '#1B2B34', '#66787F', '#DFE6EA', '#AEBBC2', '#FFFFFF'

TOPO_COLOR  = {'circular': '#0072B2', 'grid': '#E69F00',        # Okabe-Ito
               'spiral':   '#009E73', 'concentric': '#CC79A7'}
TOPO_MARKER = {'circular': 'o', 'grid': 's', 'spiral': '^', 'concentric': 'D'}
TOPO_LABEL  = {'circular': 'circular (UCA)', 'grid': 'grilla',
               'spiral': 'espiral', 'concentric': 'concentrica'}

M_COLOR  = {6: '#9FC2D4', 8: '#4A88A6', 12: '#164A60'}          # secuencial frio
RT_COLOR = {0.3: '#F0C480', 0.5: '#D8823F', 0.8: '#9B4318'}     # secuencial calido
RT_INK   = {0.3: '#A97318', 0.5: '#B0611F', 0.8: '#7C3312'}     # el mismo, legible en texto

PROC_COLOR = {'NM_MVDR': '#164A60', 'SOUDEN_ORACLE_SCM': '#AAB6BD'}   # relleno
PROC_INK   = {'NM_MVDR': '#164A60', 'SOUDEN_ORACLE_SCM': '#5C6B73'}   # texto (contraste)
PROC_LABEL = {'NM_MVDR': 'NM-MVDR (mascara DTLN)',
              'SOUDEN_ORACLE_SCM': 'Souden oracle (techo)'}

CMAP_SEQ = LinearSegmentedColormap.from_list(
    'seq_cool', ['#F5F9FB', '#C3DCE7', '#7FB2C8', '#3A7E9C', '#123C4F'])

# ------------------------------ RCPARAMS ------------------------------
plt.rcParams.update({
    'figure.dpi': 108, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': PAPER, 'axes.facecolor': PAPER,
    'font.size': 10.0, 'font.family': 'DejaVu Sans',
    'text.color': INK, 'axes.labelcolor': INK,
    'axes.titlesize': 10.8, 'axes.titleweight': 'semibold',
    'axes.titlepad': 8, 'axes.titlelocation': 'left',
    'axes.labelsize': 9.6, 'axes.labelpad': 6,
    'axes.edgecolor': EDGE, 'axes.linewidth': 0.9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': GRIDC, 'grid.linewidth': 0.8, 'grid.alpha': 1.0,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelsize': 9.0, 'ytick.labelsize': 9.0,
    'xtick.major.size': 3.0, 'ytick.major.size': 3.0, 'xtick.major.width': 0.8,
    'legend.frameon': False, 'legend.fontsize': 8.8, 'legend.title_fontsize': 8.8,
    'figure.titlesize': 13, 'figure.titleweight': 'semibold',
    'axes.prop_cycle': cycler(color=list(TOPO_COLOR.values())),
})

# ------------------------------ HELPERS -------------------------------
import textwrap

def head(fig, title, subtitle=None, wrap=None):
    """Titulo + bajada alineados a la izquierda. Devuelve el `top` que hay que
    pasarle a tight_layout(rect=...) para que los ejes no se le suban encima.
    El desplazamiento se calcula en PULGADAS y se normaliza por la altura de la
    figura: si no, en figuras altas el subtitulo se pega al titulo."""
    h = fig.get_figheight()
    wrap = wrap or int(fig.get_figwidth() * 13.5)
    fig.suptitle(title, x=0.006, y=1 - 0.06 / h, ha='left', va='top')
    n = 0
    if subtitle:
        txt = textwrap.fill(subtitle, wrap)
        n = txt.count('\n') + 1
        fig.text(0.006, 1 - 0.32 / h, txt, ha='left', va='top',
                 fontsize=9.2, color=MUTED, linespacing=1.35)
    return 1 - (0.42 + 0.17 * n) / h

def head_legend(fig, ax, ncols=2):
    """Mueve la leyenda de `ax` al encabezado de la figura (arriba a la derecha),
    donde no puede taparle datos a ningun panel."""
    hs, ls = ax.get_legend_handles_labels()
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    fig.legend(hs, ls, loc='upper right',
               bbox_to_anchor=(0.995, 1 - 0.05 / fig.get_figheight()),
               ncols=ncols, frameon=False, handletextpad=0.5, columnspacing=1.4)

def row_labels(fig, axes, labels, colors):
    """Etiqueta de fila rotada, al margen izquierdo. Se llama DESPUES de
    tight_layout: usa la posicion final de los ejes."""
    for r, (lab, col) in enumerate(zip(labels, colors)):
        bb = axes[r][0].get_position()
        fig.text(0.004, (bb.y0 + bb.y1) / 2, lab, rotation=90, va='center', ha='left',
                 fontsize=9.4, color=col, fontweight='semibold')

def only_grid(ax, axis='y'):
    ax.grid(False)
    ax.grid(True, axis=axis, color=GRIDC, lw=0.8)

def bar_value_labels(ax, bars, fmt='{:+.2f}', horizontal=False, color=INK):
    for b in bars:
        if horizontal:
            w = b.get_width()
            ax.annotate(fmt.format(w), (w, b.get_y() + b.get_height()/2),
                        xytext=(4 if w >= 0 else -4, 0), textcoords='offset points',
                        va='center', ha='left' if w >= 0 else 'right',
                        fontsize=8.4, color=color)
        else:
            h = b.get_height()
            ax.annotate(fmt.format(h), (b.get_x() + b.get_width()/2, h),
                        xytext=(0, 3 if h >= 0 else -3), textcoords='offset points',
                        ha='center', va='bottom' if h >= 0 else 'top',
                        fontsize=8.4, color=color)

def show_table(dframe, title=None, fmt='{:.2f}'):
    if title:
        print(f'\n{title}\n' + '-' * max(len(title), 12))
    with pd.option_context('display.float_format', fmt.format,
                           'display.width', 200, 'display.max_columns', 50):
        print(dframe.to_string())

print('[*] sistema visual cargado')

---
# Parte 1 — Condiciones de la simulación

## 1. Diseño factorial y cobertura

Las constantes de abajo son las **mismas** del notebook de corrida: describen la física
del experimento y no se usan para recalcular nada, solo para dibujar y contextualizar.
La celda siguiente **verifica contra el dataset** que lo declarado acá sea lo que
efectivamente se corrió (si no coincide, aborta).

**Superficie equivalente.** La referencia es el **círculo (UCA) de 15 cm**,
`A_ref = π(D/2)² ≈ 176,7 cm²`. `circular`, `spiral` y `concentric` quedan inscriptas en
ese círculo; la **grilla** ya no se inscribe sino que se iguala **por área**
(`area_mode='span'`), lo que la libera de ser `n×n` y permite 2×3, 2×4, 3×4.

**Bafle.** El arreglo es planar en XY y se apoya **flush sobre el piso** (0,5 cm), que
actúa de bafle horizontal — un dispositivo sobre una mesa, con broadside hacia arriba.
El piso no es rígido (mismo α que fija el RT60), así que el bafle es **parcial**.

**Micrófono de referencia** (`ref_mic_mode='centroid'`): el más cercano al centro
geométrico. Es a la vez el punto de escucha sobre el que proyectan los beamformers de
Souden **y** el canal contra el que se miden baseline y referencias — un solo punto de
escucha, no dos ([[metrics-ref-mic-mismatch]]).

In [ ]:
# ----------------------------- ARREGLO --------------------------------
M_LIST     = [6, 8, 12]
DIAMETER   = 0.15                              # [m] circulo de referencia
A_REF      = np.pi * (DIAMETER / 2.0) ** 2     # [m^2] superficie que ocupan TODAS
TOPOLOGIES = ['circular', 'grid', 'spiral', 'concentric']
TOPOLOGY_KWARGS = {'grid': {'area_mode': 'span'}}
REF_MIC_MODE = 'centroid'
C_SOUND = 343.0

def topo_kwargs_for(topo):
    return dict(TOPOLOGY_KWARGS.get(topo, {}))

# ------------------------------ SALAS ---------------------------------
ROOM_PROFILES = {0.30: np.array([6.0, 7.0, 2.8]),    # oficina
                 0.50: np.array([7.0, 8.0, 3.0]),    # aula
                 0.80: np.array([9.0, 11.0, 3.5])}   # salon
ROOM_LABEL = {0.30: 'oficina', 0.50: 'aula', 0.80: 'salon'}

BAFFLE_HEIGHT = 0.005     # [m] arreglo flush sobre el piso
WALL_MARGIN   = 0.30      # [m] aire minimo contra cada pared
ARRAY_CENTER_MAP = {0.30: np.array([3.0, 2.5, BAFFLE_HEIGHT]),
                    0.50: np.array([3.5, 3.0, BAFFLE_HEIGHT]),
                    0.80: np.array([4.5, 3.5, BAFFLE_HEIGHT])}

# ------------------------- ESCENAS ALEATORIAS -------------------------
TORUS_R_MAJOR, TORUS_R_TUBE = 1.10, 0.45   # toroide de locutores, tangente al piso
ISIR_DB, DURATION, SNR_DB   = 0, 15, 60.0  # interferencia tan fuerte como el target

# ------------------- VERIFICACION contra el dataset -------------------
assert sorted(df['topology'].unique()) == sorted(TOPOLOGIES), 'topologias != dataset'
assert sorted(df['M'].unique()) == sorted(M_LIST),            'M != dataset'
assert sorted(df['rt60'].unique()) == sorted(ROOM_PROFILES),  'rt60 != dataset'
assert np.isclose(df['diameter'].unique(), DIAMETER).all(),   'diametro != dataset'

SCENE_IDS = sorted(df['interf_scenario'].unique())
PROCS     = [p for p in ['NM_MVDR', 'SOUDEN_ORACLE_SCM'] if p in df['processor'].unique()]
N_SCENES  = len(SCENE_IDS)

# Geometria REAL de cada escena, tal como la registro el motor (una fila por escena).
SCN = (df.groupby('interf_scenario')[
           ['source_azimuth_deg', 'source_elevation_deg', 'source_slant_m', 'source_height_m',
            'interf_azimuth_deg', 'interf_elevation_deg', 'interf_slant_m']]
         .mean().loc[SCENE_IDS])      # ptp intra-escena < 0.5 deg: promediar es inocuo

print(f'Topologias  : {TOPOLOGIES}')
print(f'M           : {M_LIST}   |  superficie comun A_ref = {A_REF*1e4:.1f} cm2 (circulo de {DIAMETER*100:.0f} cm)')
print(f'Salas       : ' + ' | '.join(f'RT={k:.2f}s {ROOM_LABEL[k]} {tuple(v)}' for k, v in ROOM_PROFILES.items()))
print(f'Escenas     : {N_SCENES} pares (target, interferencia) aleatorios, compartidos por las 3 salas')
print(f'Procesadores: ' + ' | '.join(PROC_LABEL[p] for p in PROCS))
print(f'Senal       : {DURATION}s, SNR={SNR_DB:.0f} dB, iSIR={ISIR_DB} dB, mic de ref = {REF_MIC_MODE}')
print(f'\nTarget   : toroide R={TORUS_R_MAJOR} m / r={TORUS_R_TUBE} m  ->  '
      f'elevacion {SCN.source_elevation_deg.min():.1f}-{SCN.source_elevation_deg.max():.1f} deg, '
      f'dist {SCN.source_slant_m.min():.2f}-{SCN.source_slant_m.max():.2f} m, '
      f'altura {SCN.source_height_m.min():.2f}-{SCN.source_height_m.max():.2f} m')
print(f'Interfer.: uniforme en el volumen de la sala  ->  '
      f'elevacion {SCN.interf_elevation_deg.min():.1f}-{SCN.interf_elevation_deg.max():.1f} deg, '
      f'dist {SCN.interf_slant_m.min():.2f}-{SCN.interf_slant_m.max():.2f} m')

In [ ]:
# Cobertura del diseno factorial: cuantas filas cayeron en cada celda
# (esperado: N_SCENES x len(PROCS) por cada (topologia, M, rt60)).
cov = (df.pivot_table(index=['topology', 'M'], columns='rt60',
                      values='exec_time_s', aggfunc='size')
         .reindex(pd.MultiIndex.from_product([TOPOLOGIES, M_LIST], names=['topology', 'M'])))
cov.columns = [f'RT={rt:.2f}s' for rt in cov.columns]   # str: si no, float_format las redondea
esperado = N_SCENES * len(PROCS)
show_table(cov, f'Filas por celda (topologia x M x RT60) — esperado {esperado}', '{:.0f}')
print(f'\nCompleto: {bool((cov == esperado).all().all())}   |   total {len(df):,} filas '
      f'= {len(TOPOLOGIES)}x{len(M_LIST)}x{len(ROOM_PROFILES)}x{N_SCENES}x{len(PROCS)}')
print(f'Escenas fisicas simuladas (RIRs): {len(df)//len(PROCS):,}')

## 2. Geometrías — misma superficie, distinto reparto

Las cuatro topologías ocupan **la misma área**; lo que cambia es **cómo reparten** los
micrófonos dentro de ella. Ese reparto fija dos cosas que el beamformer paga directo:

- **`d_min`** — la separación mínima entre micrófonos, que fija la **frecuencia de
  aliasing espacial** `f_alias = c / 2·d_min`. Por encima de `f_alias` aparecen lóbulos
  de rejilla y el filtro puede confundir la dirección de la interferencia con la del
  target.
- **apertura máxima** — la distancia entre los dos micrófonos más lejanos, que fija la
  resolución angular en **baja** frecuencia.

Las dos tiran para lados opuestos con área fija: repartir hacia el borde (circular) da
apertura pero deja huecos; concentrar (grilla, concéntrica) sube `f_alias` y sacrifica
apertura. Ese es todo el compromiso que este barrido pone a prueba.

In [ ]:
rows = []
for M in M_LIST:
    for topo in TOPOLOGIES:
        c  = generate_array_coords(topo, M, DIAMETER, **topo_kwargs_for(topo))[:, :2]
        dd = np.linalg.norm(c[:, None, :] - c[None, :, :], axis=-1)
        d_min = dd[dd > 0].min()
        span  = c.max(axis=0) - c.min(axis=0)
        # Area ocupada: rectangulo abarcado (grilla, equivalente por area) vs disco
        # de referencia (topologias inscriptas, cuya area es la del circulo).
        area = span[0] * span[1] if topo == 'grid' else A_REF
        rows.append({'M': M, 'topologia': topo, 'ref_mic': select_reference_mic(c),
                     'd_min_cm': d_min * 100, 'f_alias_kHz': C_SOUND / (2 * d_min) / 1e3,
                     'apertura_cm': dd.max() * 100,
                     'span_cm': f'{span[0]*100:.1f} x {span[1]*100:.1f}',
                     'area_cm2': area * 1e4})
GEO = pd.DataFrame(rows)

show_table(GEO.set_index(['M', 'topologia']),
           f'Propiedades del arreglo — A_ref = {A_REF*1e4:.1f} cm2 para todas')

# --- Los dos ejes del compromiso, uno al lado del otro ---
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.7))
for ax, (col, ylab, ttl) in zip(axes, [
        ('f_alias_kHz', 'f_alias = c / 2·d_min  [kHz]', 'Aliasing espacial (alta frecuencia)'),
        ('apertura_cm', 'apertura maxima  [cm]',        'Apertura (resolucion en baja frecuencia)')]):
    for topo in TOPOLOGIES:
        s = GEO[GEO.topologia == topo].set_index('M')[col]
        ax.plot(s.index, s.values, marker=TOPO_MARKER[topo], color=TOPO_COLOR[topo],
                lw=2.0, ms=7, mec='white', mew=1.2, label=TOPO_LABEL[topo])
    ax.set(xlabel='M (microfonos)', ylabel=ylab, title=ttl, xticks=M_LIST)
    only_grid(ax)
axes[0].axhline(8.0, color=MUTED, ls=':', lw=1.2)
axes[0].annotate('Nyquist de la senal (8 kHz)', (M_LIST[0], 8.0), xytext=(2, 5),
                 textcoords='offset points', fontsize=8.2, color=MUTED)
top = head(fig, 'Con área fija, apertura y f_alias tiran para lados opuestos',
           'Cada topología elige un punto del compromiso; el barrido mide cuál paga mejor.')
head_legend(fig, axes[1], ncols=4)
fig.tight_layout(rect=(0, 0, 1, top)); plt.show()

In [ ]:
# ============ GALERIA: una fila por M, una columna por topologia ============
import matplotlib.patches as mpatches

fig, axes = plt.subplots(len(M_LIST), len(TOPOLOGIES),
                         figsize=(3.05 * len(TOPOLOGIES), 3.25 * len(M_LIST)))
axes = np.atleast_2d(axes)
for r, M in enumerate(M_LIST):
    for cix, topo in enumerate(TOPOLOGIES):
        ax  = axes[r, cix]
        col = TOPO_COLOR[topo]
        c   = generate_array_coords(topo, M, DIAMETER, **topo_kwargs_for(topo))[:, :2]
        ref = select_reference_mic(c)
        g   = GEO[(GEO.M == M) & (GEO.topologia == topo)].iloc[0]

        # circulo de referencia (A_ref) + rectangulo equivalente por area (grilla)
        ax.add_patch(mpatches.Circle((0, 0), DIAMETER / 2, ec=EDGE, fc='#F7FAFB',
                                     ls=(0, (4, 3)), lw=1.1, zorder=0))
        if topo == 'grid':
            lo, hi = c.min(axis=0), c.max(axis=0)
            ax.add_patch(mpatches.Rectangle(lo, *(hi - lo), ec=col, fc=col,
                                            alpha=0.10, lw=1.1, ls=(0, (4, 3)), zorder=0))
        # apertura maxima, como segmento tenue
        dd = np.linalg.norm(c[:, None, :] - c[None, :, :], axis=-1)
        i, j = np.unravel_index(np.argmax(dd), dd.shape)
        ax.plot(*zip(c[i], c[j]), color=col, lw=0.9, alpha=0.35, zorder=1)

        ax.scatter(c[:, 0], c[:, 1], s=62, c=col, marker=TOPO_MARKER[topo],
                   ec='white', lw=1.1, zorder=3)
        ax.scatter(*c[ref], s=230, facecolors='none', edgecolors=INK, lw=1.6, zorder=4)
        ax.annotate('ref', c[ref], xytext=(0, -16), textcoords='offset points',
                    ha='center', fontsize=7.6, color=INK)

        lim = max(DIAMETER / 2, np.abs(c).max()) * 1.32
        ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xticks=[], yticks=[])
        ax.set_aspect('equal'); ax.grid(False)
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.set_title(f'{TOPO_LABEL[topo]}  ·  M={M}', color=col, fontsize=10.2)
        ax.text(0.5, -0.055, f'd_min {g.d_min_cm:.1f} cm  ·  f_alias {g.f_alias_kHz:.1f} kHz\n'
                             f'apertura {g.apertura_cm:.1f} cm  ·  {g.area_cm2:.0f} cm²',
                transform=ax.transAxes, ha='center', va='top', fontsize=7.8, color=MUTED)

top = head(fig, 'Las cuatro topologías, con la geometría exacta que arma el motor',
           f'Círculo punteado = superficie de referencia ({A_REF*1e4:.0f} cm²). La grilla puede '
           'salirse de él: se iguala por área, no por inscripción. El círculo negro marca el '
           'micrófono de referencia (el más cercano al centroide); la línea fina es la apertura máxima.')
fig.tight_layout(rect=(0, 0, 1, top), h_pad=2.8); plt.show()

## 3. La escena: sala, bafle, toroide de locutores y nube de interferencias

Cada una de las **60 escenas** es un par *(target, interferencia)* sorteado una sola vez
y **reutilizado idéntico** en las 3 salas × 4 topologías × 3 valores de M. Eso convierte
al barrido en un **diseño pareado**: cualquier diferencia entre topologías se mide sobre
exactamente la misma física, no sobre sorteos distintos.

- **Target — toroide apoyado sobre el piso.** No es un domo (que pondría locutores en el
  cenit y otros pegados al plano) sino un **anillo con espesor**: tubo de radio 45 cm
  tangente al piso, círculo generador a 1,10 m del centro. Modela gente hablando
  *alrededor* del dispositivo, con la boca a ~50 cm sobre el plano del arreglo, y da
  **elevaciones acotadas y realistas (≈ 5–43°)** en vez de todo el hemisferio.
- **Interferencia — uniforme en el volumen de la sala.** Sortear (az, el, dist) por
  separado *no* llena la sala: deja las fuentes en un cascarón y las esquinas vacías.
  Acá se sortea el **punto** uniforme en el volumen útil y recién después se pasa a
  coordenadas esféricas.

Las posiciones que se dibujan salen del **dataset** (columnas `source_*` / `interf_*`
que escribió el motor), no de volver a sortear con la semilla.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (habilita la proyeccion 3d)

RT_SHOW, TOPO_SHOW, M_SHOW = 0.50, 'grid', M_LIST[-1]
ac   = np.asarray(ARRAY_CENTER_MAP[RT_SHOW], float).copy()
room = np.asarray(ROOM_PROFILES[RT_SHOW],   float).copy()
mic  = generate_array_coords(TOPO_SHOW, M_SHOW, DIAMETER, **topo_kwargs_for(TOPO_SHOW)) + ac
ref  = select_reference_mic(mic)

tgt = np.array([place_spherical(*SCN.loc[s, ['source_azimuth_deg', 'source_elevation_deg',
                                             'source_slant_m']], ac) for s in SCENE_IDS])
itf = np.array([place_spherical(*SCN.loc[s, ['interf_azimuth_deg', 'interf_elevation_deg',
                                             'interf_slant_m']], ac) for s in SCENE_IDS])

C_TGT, C_ITF, C_ARR = '#009E73', '#CC5B32', '#164A60'

def _room_box(ax, room, **kw):
    x, y, z = room
    P = np.array([[0,0,0],[x,0,0],[x,y,0],[0,y,0],[0,0,z],[x,0,z],[x,y,z],[0,y,z]], float)
    for a, b in [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]:
        ax.plot(*zip(P[a], P[b]), **kw)

def _torus(ax, center, R, r, **kw):
    u = np.linspace(0, 2*np.pi, 72); v = np.linspace(0, 2*np.pi, 28)
    U, V = np.meshgrid(u, v)
    ax.plot_surface(center[0] + (R + r*np.cos(V))*np.sin(U),
                    center[1] + (R + r*np.cos(V))*np.cos(U),
                    r + r*np.sin(V), **kw)

fig = plt.figure(figsize=(11.5, 6.8))
ax = fig.add_subplot(111, projection='3d')
_room_box(ax, room, color=EDGE, ls='--', lw=0.9)
# bafle: el piso de la sala
ax.plot_trisurf([0, room[0], room[0], 0], [0, 0, room[1], room[1]], [0]*4,
                color=C_ARR, alpha=0.07, linewidth=0)
_torus(ax, ac, TORUS_R_MAJOR, TORUS_R_TUBE, color=C_TGT, alpha=0.09,
       linewidth=0, antialiased=True, shade=False)

ax.scatter(*itf.T, c=C_ITF, marker='v', s=46, depthshade=False, alpha=0.85,
           ec='white', lw=0.5, label=f'interferencia ({N_SCENES}, uniforme en el volumen)')
ax.scatter(*tgt.T, c=C_TGT, marker='*', s=135, depthshade=False,
           ec='white', lw=0.5, label=f'target ({N_SCENES}, toroide sobre el piso)')
ax.scatter(*mic.T, c=C_ARR, marker='x', s=34, depthshade=False,
           label=f'arreglo ({TOPO_LABEL[TOPO_SHOW]}, M={M_SHOW})')
ax.scatter(*mic[ref], facecolors='none', edgecolors=INK, s=110, lw=1.6,
           depthshade=False, label=f'microfono de referencia (#{ref})')

ax.set(xlabel='X [m]', ylabel='Y [m]', zlabel='Z (altura) [m]',
       xlim=(0, room[0]), ylim=(0, room[1]), zlim=(0, room[2]))
for pane in (ax.xaxis, ax.yaxis, ax.zaxis):
    pane.pane.set_facecolor('white'); pane.pane.set_edgecolor(GRIDC); pane.pane.set_alpha(1.0)
ax._axis3don = True
ax.grid(True, color=GRIDC, lw=0.6)
ax.view_init(elev=20, azim=-62)
ax.legend(loc='upper left', bbox_to_anchor=(0.03, 0.93), fontsize=8.6)
# OJO: set_box_aspect MUTA el array que recibe -> pasar SIEMPRE una tupla.
# `zoom` agranda el cuerpo 3D dentro del eje sin deformar la escala: la sala es
# ancha y baja, y sin zoom queda una franja chica perdida en medio de la figura.
ax.set_box_aspect(tuple(room), zoom=1.16)
head(fig, f'Escena 3D — sala {ROOM_LABEL[RT_SHOW]} (RT60 = {RT_SHOW:.2f} s)',
     'Las 60 escenas superpuestas. Los locutores viven en el toroide verde; las '
     'interferencias, en cualquier parte del volumen útil de la sala.')
# tight_layout no maneja bien los ejes 3D: el margen se ajusta a mano.
fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=0.90)
plt.show()

In [ ]:
# =============== De donde vienen realmente target e interferencia ===============
r_h_t = SCN.source_slant_m * np.cos(np.deg2rad(SCN.source_elevation_deg))
z_t   = SCN.source_slant_m * np.sin(np.deg2rad(SCN.source_elevation_deg)) + BAFFLE_HEIGHT
r_h_i = SCN.interf_slant_m * np.cos(np.deg2rad(SCN.interf_elevation_deg))
z_i   = SCN.interf_slant_m * np.sin(np.deg2rad(SCN.interf_elevation_deg)) + BAFFLE_HEIGHT

fig = plt.figure(figsize=(13.0, 7.4))
gs  = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.28)

# (a) corte vertical: el toroide de perfil
ax = fig.add_subplot(gs[0, 0])
ax.scatter(r_h_i, z_i, s=26, c=C_ITF, alpha=0.55, ec='white', lw=0.4, label='interferencia')
ax.scatter(r_h_t, z_t, s=48, c=C_TGT, marker='*', ec='white', lw=0.4, label='target')
ax.add_patch(plt.Circle((TORUS_R_MAJOR, TORUS_R_TUBE), TORUS_R_TUBE,
                        ec=C_TGT, fc='none', ls=(0, (4, 3)), lw=1.2))
ax.axhline(0, color=INK, lw=1.0)
ax.scatter([0], [BAFFLE_HEIGHT], marker='x', c=C_ARR, s=44)
ax.set(xlabel='distancia horizontal al arreglo [m]', ylabel='altura sobre el piso [m]',
       title='(a) Corte vertical de la escena', xlim=(-0.15, 5.2))
ax.set_aspect('equal'); ax.grid(True, color=GRIDC, lw=0.8)
ax.legend(loc='upper right', frameon=True, framealpha=0.92, edgecolor='none',
          facecolor=PAPER, fontsize=8.4)

# (b) elevacion: el eje que peor discrimina un arreglo PLANAR
ax = fig.add_subplot(gs[0, 1])
bins = np.arange(0, 75, 5)
ax.hist(SCN.interf_elevation_deg, bins=bins, color=C_ITF, alpha=0.55, label='interferencia')
ax.hist(SCN.source_elevation_deg, bins=bins, color=C_TGT, alpha=0.85, label='target')
ax.set(xlabel='elevacion sobre el plano del arreglo [deg]', ylabel='escenas',
       title='(b) Elevacion — eje critico del analisis')
only_grid(ax); ax.legend()

# (c) distancia
ax = fig.add_subplot(gs[0, 2])
ax.hist(SCN.interf_slant_m, bins=np.arange(0, 5.5, 0.35), color=C_ITF, alpha=0.55,
        label='interferencia')
ax.hist(SCN.source_slant_m, bins=np.arange(0, 5.5, 0.35), color=C_TGT, alpha=0.85,
        label='target')
ax.set(xlabel='distancia al centro del arreglo [m]', ylabel='escenas',
       title='(c) Distancia — el target siempre esta mas cerca')
only_grid(ax); ax.legend()

# (d) azimut: cobertura de los 360 deg
ax = fig.add_subplot(gs[1, 0], projection='polar')
ax.scatter(np.deg2rad(SCN.interf_azimuth_deg), SCN.interf_slant_m, s=26, c=C_ITF,
           alpha=0.6, ec='white', lw=0.4)
ax.scatter(np.deg2rad(SCN.source_azimuth_deg), SCN.source_slant_m, s=52, c=C_TGT,
           marker='*', ec='white', lw=0.4)
ax.set_title('(d) Azimut x distancia [m]', pad=14)
ax.set_theta_zero_location('E'); ax.grid(color=GRIDC)
ax.tick_params(labelsize=7.6)

# (e) separacion angular target-interferencia: el driver #1 del SIR alcanzable
ax = fig.add_subplot(gs[1, 1])
d_az = np.abs(SCN.source_azimuth_deg - SCN.interf_azimuth_deg) % 360
d_az = np.minimum(d_az, 360 - d_az)
ax.hist(d_az, bins=np.arange(0, 190, 15), color=C_ARR, alpha=0.85)
ax.axvline(np.median(d_az), color=C_ITF, lw=1.6, ls='--')
ax.set_ylim(0, ax.get_ylim()[1] * 1.18)
ax.annotate(f'mediana {np.median(d_az):.0f}°', (np.median(d_az), ax.get_ylim()[1]),
            xytext=(6, -11), textcoords='offset points', color=C_ITF, fontsize=8.6, va='top')
ax.set(xlabel='|Δ azimut| target vs interferencia [deg]', ylabel='escenas',
       title='(e) Separacion angular entre fuentes')
only_grid(ax)

# (f) resumen de las salas
ax = fig.add_subplot(gs[1, 2])
# Rotulos apilados ARRIBA de las salas: puestos dentro de cada rectangulo se
# pisan entre si (las tres salas comparten la esquina inferior izquierda).
for i, (rt, room_i) in enumerate(ROOM_PROFILES.items()):
    ax.add_patch(mpatches.Rectangle((0, 0), room_i[0], room_i[1], fc=RT_COLOR[rt],
                                    alpha=0.16, ec=RT_COLOR[rt], lw=1.6))
    ax.scatter(*ARRAY_CENTER_MAP[rt][:2], marker='x', s=52, c=RT_COLOR[rt])
    ax.annotate(f'RT={rt:.2f} s · {ROOM_LABEL[rt]} · '
                f'{room_i[0]:.0f}×{room_i[1]:.0f}×{room_i[2]:.1f} m',
                (0.05, 14.7 - 0.95 * i), ha='left', va='center',
                fontsize=7.6, color=RT_INK[rt], fontweight='semibold')
ax.set(xlim=(-0.4, 9.6), ylim=(-0.4, 15.3), xlabel='X [m]', ylabel='Y [m]',
       title='(f) Las 3 salas (planta) y el centro del arreglo')
ax.set_aspect('equal'); ax.grid(True, color=GRIDC, lw=0.8)

head(fig, 'Cómo se reparten las 60 escenas',
     'El target queda cerca y bajo (toroide); la interferencia, lejos y en cualquier '
     'dirección. Un arreglo planar solo puede separarlas por azimut y por elevación.')
plt.show()

---
# Parte 2 — Resultados

Las cuatro métricas se leen siempre como **Δ respecto del micrófono de referencia
crudo** (`Delta_tot_*_early`): cuánto mejora la señal *después* del beamformer contra
lo que se oiría *sin* procesar en ese mismo punto de escucha. La referencia limpia es
la componente **early** (directo + primeras reflexiones, `t_early = 50 ms`), que es lo
que un beamformer puede aspirar a recuperar — pedirle la anecoica sería pedirle que
también desreverbere.

Todo se agrega por **mediana** sobre las `60 escenas × 3 salas` (y, cuando corresponde,
× 3 valores de M).

Los dos procesadores no compiten entre sí:

- **NM-MVDR** — el sistema real, con máscara estimada por DTLN. Es el que decide.
- **Souden oracle** — mismas SCM pero calculadas con la máscara ideal. Es el **techo**
  que la topología permite alcanzar si la estimación de máscara fuera perfecta; sirve
  para saber si una diferencia entre topologías es *geometría* o *estimación*.

In [ ]:
REF     = 'early'
METRICS = ['PESQ', 'STOI', 'SI-SDR', 'SIR']
MLABEL  = {'PESQ': 'Δ PESQ  [MOS-LQO]', 'STOI': 'Δ STOI  [0–1]',
           'SI-SDR': 'Δ SI-SDR  [dB]',  'SIR': 'Δ SIR  [dB]'}
MSHORT  = {'PESQ': 'Δ PESQ', 'STOI': 'Δ STOI', 'SI-SDR': 'Δ SI-SDR [dB]', 'SIR': 'Δ SIR [dB]'}
MFMT    = {'PESQ': '{:+.3f}', 'STOI': '{:+.3f}', 'SI-SDR': '{:+.2f}', 'SIR': '{:+.2f}'}

KEEP = ['topology', 'M', 'processor', 'rt60', 'interf_scenario',
        'source_elevation_deg', 'source_azimuth_deg', 'source_slant_m',
        'interf_elevation_deg', 'ref_mic_idx']
view = (df[KEEP + [f'Delta_tot_{m}_{REF}' for m in METRICS]]
          .rename(columns={f'Delta_tot_{m}_{REF}': m for m in METRICS})
          .copy())
view['topology'] = pd.Categorical(view['topology'], categories=TOPOLOGIES, ordered=True)

# Bins de elevacion del target (el toroide barre ~5-43 deg de forma controlada).
EL_EDGES = np.arange(0, 50, 7.5)
view['el_mid'] = (pd.cut(view['source_elevation_deg'], EL_EDGES)
                    .map(lambda b: b.mid if pd.notna(b) else np.nan).astype(float))

def med(dframe, by):
    """Mediana de las 4 metricas agrupando por `by` (el SIR trae NaN: se dropean)."""
    return dframe.groupby(by, observed=True)[METRICS].median()

print('NaN por metrica (el SIR es infinito/indefinido en escenas casi perfectas):')
print(view[METRICS].isna().sum().to_string())
print(f'\nfilas utiles: {len(view):,}')

## 4. Ranking global por topología

Mediana sobre **las 540 escenas** de cada topología (60 escenas × 3 salas × 3 valores de
M). Es el número que responde la pregunta del barrido.

In [ ]:
for proc in PROCS:
    t = med(view[view.processor == proc], 'topology').reindex(TOPOLOGIES)
    t.loc['—— mejor vs peor'] = t.max() - t.min()
    show_table(t, f'{PROC_LABEL[proc]} — mediana de Δ vs {REF}', '{:+.3f}')

fig, axes = plt.subplots(1, 4, figsize=(14.0, 4.3))
for ax, metric in zip(axes, METRICS):
    piv = (med(view, ['topology', 'processor'])[metric]
             .unstack('processor').reindex(TOPOLOGIES))
    y = np.arange(len(TOPOLOGIES))[::-1]
    h = 0.36
    for k, proc in enumerate(PROCS):
        bars = ax.barh(y + (0.5 - k) * h, piv[proc].values, height=h,
                       color=PROC_COLOR[proc], label=PROC_LABEL[proc],
                       ec='white', lw=0.8, zorder=2)
        bar_value_labels(ax, bars, MFMT[metric], horizontal=True,
                         color=INK if proc == 'NM_MVDR' else MUTED)
    best = piv['NM_MVDR'].idxmax()
    ax.set_yticks(y, [TOPO_LABEL[t] for t in TOPOLOGIES])
    for lbl, topo in zip(ax.get_yticklabels(), TOPOLOGIES):
        lbl.set_color(TOPO_COLOR[topo])
        if topo == best:
            lbl.set_fontweight('bold')
    ax.set(xlabel=MLABEL[metric], title=metric)
    ax.set_xlim(0, ax.get_xlim()[1] * 1.22)
    ax.grid(False); ax.grid(True, axis='x', color=GRIDC, lw=0.8)
    ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)
top = head(fig, 'Grid y circular empatan arriba; la espiral es la única que pierde claro',
           'Mediana sobre 540 escenas por topología. La barra oscura es el sistema real; la gris, '
           'el techo con máscara oracle. El orden se repite en ambos: no es artefacto de la máscara.')
head_legend(fig, axes[0], ncols=2)
fig.tight_layout(rect=(0, 0, 1, top)); plt.show()

### 4.1 ¿La diferencia entre topologías es real o es ruido?

El diseño es **pareado**: la misma escena (mismo `scene_id`, misma sala, mismo M) se
corrió con las cuatro topologías. Eso permite comparar cada topología contra la mejor
*escena por escena* y preguntar si la diferencia sobrevive al test de rangos con signo
de Wilcoxon — y, más importante, **cuánto pesa** esa diferencia frente a la dispersión
que produce la propia escena.

In [ ]:
from scipy.stats import wilcoxon

def paired(metric, proc):
    """Matriz escena x topologia (mismo scene_id, rt60, M) para comparar pareado."""
    return (view[view.processor == proc]
            .pivot_table(index=['interf_scenario', 'rt60', 'M'], columns='topology',
                         values=metric, observed=True)
            .dropna())

rows = []
for proc in PROCS:
    for metric in METRICS:
        P = paired(metric, proc)
        best = P.median().idxmax()
        for topo in TOPOLOGIES:
            if topo == best:
                rows.append({'procesador': proc, 'metrica': metric, 'topologia': topo,
                             'delta_vs_mejor': 0.0, 'p_wilcoxon': np.nan,
                             'IQR_entre_escenas': P[topo].quantile(.75) - P[topo].quantile(.25)})
                continue
            d = (P[topo] - P[best]).dropna()
            p = wilcoxon(d, alternative='two-sided').pvalue if d.abs().sum() > 0 else 1.0
            rows.append({'procesador': proc, 'metrica': metric, 'topologia': topo,
                         'delta_vs_mejor': float(np.median(d)), 'p_wilcoxon': p,
                         'IQR_entre_escenas': P[topo].quantile(.75) - P[topo].quantile(.25)})
STAT = pd.DataFrame(rows)
STAT['|delta| / IQR'] = (STAT.delta_vs_mejor.abs() / STAT.IQR_entre_escenas)
STAT['significativo'] = np.where(STAT.p_wilcoxon.isna(), '— (mejor)',
                          np.where(STAT.p_wilcoxon < 0.01, 'si (p<0.01)',
                          np.where(STAT.p_wilcoxon < 0.05, 'si (p<0.05)', 'no')))

for proc in PROCS:
    s = (STAT[STAT.procesador == proc]
         .pivot_table(index='topologia', columns='metrica',
                      values=['delta_vs_mejor', '|delta| / IQR'], observed=True)
         .reindex(TOPOLOGIES))
    s = s.reindex(columns=pd.MultiIndex.from_product([['delta_vs_mejor', '|delta| / IQR'],
                                                      METRICS]))
    show_table(s, f'{PROC_LABEL[proc]} — mediana pareada vs la MEJOR topologia', '{:+.3f}')

show_table(STAT[STAT.procesador == 'NM_MVDR']
             .pivot(index='topologia', columns='metrica', values='significativo')
             .reindex(TOPOLOGIES)[METRICS],
           'NM-MVDR — ¿la diferencia vs la mejor topologia es estadisticamente significativa?',
           '{}')

_peor = STAT[(STAT.procesador == 'NM_MVDR') & (STAT.metrica == 'SIR')].set_index('topologia')
print('\nLectura: con N ~ 540 pares, Wilcoxon declara significativo casi todo -- incluso')
print('diferencias de 0.001 en STOI. Lo que decide si la topologia IMPORTA no es la p, sino')
print('|delta| / IQR: cuanto pesa el cambio de topologia frente a la dispersion que ya')
print('introduce la escena. Ahi el cuadro cambia:')
print(f'  - STOI y SI-SDR: |delta|/IQR <= {STAT[STAT.metrica.isin(["STOI","SI-SDR"])]["|delta| / IQR"].max():.2f}'
      '  -> la topologia es irrelevante.')
print(f'  - SIR: la espiral pierde {abs(_peor.loc["spiral","delta_vs_mejor"]):.2f} dB contra la grilla, '
      f'= {_peor.loc["spiral","|delta| / IQR"]:.0%} del IQR entre escenas')
print('    -> ahi si hay una diferencia que se puede oir. El resto empata.')

### 4.2 ¿Qué variable geométrica explica el ranking?

Con el área fija, una topología solo puede elegir dónde ponerse en el compromiso de la
sección 2. Vale entonces preguntarle a los datos **cuál de las dos variables** —
apertura máxima o `d_min` (equivalentemente `f_alias`) — predice el orden observado.
Hay 12 celdas `(topología × M)`, cada una con su geometría medida y su mediana sobre
180 escenas: alcanza para una correlación de rangos.

In [ ]:
from scipy.stats import spearmanr

GEO_K = GEO.rename(columns={'topologia': 'topology'}).set_index(['topology', 'M'])

corr_rows, cloud = [], {}
for proc in PROCS:
    cell = (view[view.processor == proc]
            .groupby(['topology', 'M'], observed=True)[METRICS].median()
            .join(GEO_K[['apertura_cm', 'd_min_cm', 'f_alias_kHz']]))
    cloud[proc] = cell.reset_index()
    for metric in METRICS:
        r = {'procesador': proc, 'metrica': metric}
        for var in ['apertura_cm', 'd_min_cm', 'f_alias_kHz']:
            r[f'rho({var})'] = spearmanr(cell[metric], cell[var]).statistic
        corr_rows.append(r)
CORR = pd.DataFrame(corr_rows).set_index(['procesador', 'metrica'])
show_table(CORR, 'Spearman entre la mediana de cada celda (topologia x M) y su geometria',
           '{:+.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.4))
for ax, var, name, xlab in zip(axes, ['apertura_cm', 'f_alias_kHz'],
                               ['apertura', 'f_alias'],
                               ['apertura maxima [cm]', 'f_alias = c / 2·d_min  [kHz]']):
    cell = cloud['NM_MVDR']
    # Tendencia global. No se unen los puntos de una misma topologia: eso sugeriria
    # una relacion DENTRO de la topologia, que no es lo que el panel muestra.
    xs = np.log10(cell[var]) if var == 'f_alias_kHz' else cell[var]
    b, a = np.polyfit(xs, cell['SIR'], 1)
    xg = np.linspace(xs.min(), xs.max(), 50)
    ax.plot(10 ** xg if var == 'f_alias_kHz' else xg, a + b * xg,
            color=MUTED, ls=(0, (5, 4)), lw=1.2, zorder=1)
    for topo in TOPOLOGIES:
        sub_c = cell[cell.topology == topo]
        ax.scatter(sub_c[var], sub_c['SIR'], s=40 + 13 * sub_c['M'], color=TOPO_COLOR[topo],
                   marker=TOPO_MARKER[topo], ec='white', lw=1.1, zorder=3,
                   label=TOPO_LABEL[topo])
        for _, row in sub_c.iterrows():
            ax.annotate(f'M={int(row.M)}', (row[var], row['SIR']), xytext=(9, -3),
                        textcoords='offset points', fontsize=7.4, color=MUTED)
    rho = spearmanr(cell['SIR'], cell[var]).statistic
    ax.set(xlabel=xlab, ylabel=MSHORT['SIR'],
           title=f'Δ SIR vs {name}   (Spearman ρ = {rho:+.2f})')
    only_grid(ax)
    if var == 'f_alias_kHz':
        ax.set_xscale('log')
top = head(fig, 'Lo que compra rechazo de interferencia es la apertura, no la densidad',
           'Cada punto es una celda (topología × M), con el tamaño creciendo con M. A M fijo el '
           'orden del SIR reproduce el de la apertura (ρ ≈ +0,95). La f_alias no explica nada: la '
           'espiral tiene la más alta de todas (25 kHz) y es la que peor rinde.')
head_legend(fig, axes[0], ncols=4)
fig.tight_layout(rect=(0, 0, 1, top)); plt.show()

# La rho global mezcla dos ejes (topologia y M). Para descartar que la explique M,
# se repite la correlacion DENTRO de cada M: ahi solo varia la topologia.
print('\nSIR vs apertura DENTRO de cada M (4 topologias por fila, sin el eje M de por medio):')
for proc in PROCS:
    cell = cloud[proc]
    tramo = ' | '.join(
        f'M={M}: rho={spearmanr(cell[cell.M == M]["SIR"], cell[cell.M == M].apertura_cm).statistic:+.2f}'
        for M in M_LIST)
    print(f'  {PROC_LABEL[proc]:<26} {tramo}')
print('  -> el orden por apertura se sostiene a M fijo: no es un efecto de M disfrazado.')

print('\nApertura maxima por topologia [cm]:')
show_table(GEO.pivot(index='topologia', columns='M', values='apertura_cm').reindex(TOPOLOGIES),
           None, '{:.1f}')
print()
print('La historia fisica cierra: la grilla, al liberarse de la restriccion de estar')
print('INSCRIPTA en el circulo y solo igualar el AREA, se estira hasta 20-24 cm de')
print('apertura contra los 15 cm (= el diametro) de circular y concentrica, y los ~13 cm')
print('de la espiral. Mas apertura = haz mas angosto en baja frecuencia = mejor nulo')
print('sobre la interferencia. La espiral hace lo contrario: concentra micros cerca del')
print('centro (d_min de 0.7 cm con M=12), lo que le da una f_alias altisima que NO le')
print('sirve -- la voz tiene poca energia arriba de 4 kHz -- y le cuesta la apertura.')

## 5. Efecto del número de micrófonos

`M` es el otro grado de libertad, y a diferencia de la topología **cuesta hardware**:
más canales de ADC, más MACs por frame en la FPGA. La pregunta práctica es cuánto
compra cada micrófono adicional y si alguna topología aprovecha mejor el presupuesto.

Ojo con un detalle del diseño: acá subir `M` **no agranda el arreglo**, porque el área
está fija en `A_ref`. Doce micrófonos son doce micrófonos *más juntos*, no un arreglo
más grande. Se gana promediado (más canales para estimar la SCM y más ganancia de
array) pero no apertura, y por eso la pendiente es suave — y no idéntica en las cuatro
métricas.

In [ ]:
fig, axes = plt.subplots(len(PROCS), 4, figsize=(14.0, 3.6 * len(PROCS)), squeeze=False)
for r, proc in enumerate(PROCS):
    sub = view[view.processor == proc]
    for c, metric in enumerate(METRICS):
        ax  = axes[r][c]
        piv = (sub.pivot_table(index='M', columns='topology', values=metric,
                               aggfunc='median', observed=True).reindex(M_LIST))
        for topo in TOPOLOGIES:
            ax.plot(piv.index, piv[topo].values, marker=TOPO_MARKER[topo],
                    color=TOPO_COLOR[topo], lw=1.9, ms=6.5, mec='white', mew=1.1,
                    label=TOPO_LABEL[topo])
        # Cada panel lleva su ylabel: las 4 metricas tienen escalas y unidades
        # distintas, asi que compartir la etiqueta de la columna 0 seria un error.
        ax.set(xticks=M_LIST, xlabel='M (microfonos)' if r == len(PROCS)-1 else '',
               ylabel=MSHORT[metric], title=f'{metric}' if r == 0 else '')
        only_grid(ax)
top = head(fig, 'Subir M mejora PESQ, STOI y SIR; el SI-SDR se queda quieto',
           'Mediana por M (agregando salas y escenas). Las curvas suben en paralelo: ninguna '
           'topología aprovecha mejor los micrófonos extra. El panel de SI-SDR engaña por la '
           'escala: su rango vertical entero es de 0,4 dB (ver §5.1).')
head_legend(fig, axes[0][3], ncols=4)
fig.tight_layout(rect=(0.032, 0, 1, top))
row_labels(fig, axes, [PROC_LABEL[p] for p in PROCS], [PROC_INK[p] for p in PROCS])
plt.show()

for proc in PROCS:
    piv = (view[view.processor == proc]
           .pivot_table(index='topology', columns='M', values=METRICS,
                        aggfunc='median', observed=True).reindex(TOPOLOGIES))
    show_table(piv.reindex(columns=pd.MultiIndex.from_product([METRICS, M_LIST])),
               f'{PROC_LABEL[proc]} — mediana por (topologia x M)', '{:+.3f}')

# Cuanto compra DUPLICAR los micros, frente a cuanto compra elegir bien la topologia.
sub  = view[view.processor == 'NM_MVDR']
comp = []
for metric in METRICS:
    by_m = sub.groupby('M', observed=True)[metric].median()
    by_t = sub.groupby('topology', observed=True)[metric].median()
    iqr  = sub[metric].quantile(.75) - sub[metric].quantile(.25)
    g_m, g_t = by_m[M_LIST[-1]] - by_m[M_LIST[0]], by_t.max() - by_t.min()
    comp.append({'metrica': metric, f'M {M_LIST[0]}->{M_LIST[-1]}': g_m,
                 'peor->mejor topologia': g_t, 'IQR entre escenas': iqr,
                 'M / topologia': g_m / g_t if g_t else np.nan,
                 'M / IQR  [%]': 100 * g_m / iqr})
show_table(pd.DataFrame(comp).set_index('metrica'),
           'NM-MVDR — duplicar M vs elegir la mejor topologia (ambos con el area fija)', '{:+.3f}')

_gm = comp[3][f'M {M_LIST[0]}->{M_LIST[-1]}']
print('\nLo que la tabla deja ver:')
print(f'  - Duplicar M ({M_LIST[0]} -> {M_LIST[-1]}, el doble de hardware) compra {_gm:.2f} dB de SIR: MENOS que la')
print(f'    diferencia entre la mejor y la peor topologia ({comp[3]["peor->mejor topologia"]:.2f} dB), y apenas')
print(f'    {comp[3]["M / IQR  [%]"]:.0f} % del IQR entre escenas. Con el area fija, mas micros rinden poco.')
print(f'  - El SI-SDR aparece con signo negativo ({comp[2][f"M {M_LIST[0]}->{M_LIST[-1]}"]:+.2f} dB), pero eso NO alcanza para')
print('    decir que empeora: es un vigesimo del IQR entre escenas. Se analiza en §5.1.')

### 5.1 Chequeo: ¿el SI-SDR está midiendo bien?

En el panel de arriba el SI-SDR es la única métrica que baja al subir `M`, y eso
merece una revisión antes de darlo por bueno: un −0,2 dB en la métrica que mide
fidelidad de forma de onda podría ser un efecto real (el MVDR distorsiona más al
tener más grados de libertad) o podría ser la métrica portándose mal.

Hay tres cosas que conviene separar:

1. **¿Es un artefacto del micrófono de referencia?** Con `ref_mic_mode='centroid'`,
   cambiar `M` cambia *qué* micrófono es el de referencia, y con él el baseline contra
   el que se calcula la Δ. Si el baseline mejorara con `M`, la Δ se achicaría sin que
   pase nada en la salida.
2. **¿Qué tan grande es realmente?** El diseño es pareado (misma escena con M = 6 y
   con M = 12), así que se puede medir la diferencia escena a escena y contar en qué
   fracción de escenas empeora de verdad.
3. **¿El SDR dice lo mismo?** SDR y SI-SDR miden lo mismo con una diferencia clave:
   BSS-Eval (SDR) admite un **filtro de distorsión de 512 taps** antes de comparar, o
   sea *perdona cualquier coloración lineal*; el SI-SDR solo perdona una **ganancia
   escalar**. Un beamformer es exactamente un filtro lineal, así que la brecha
   `SDR − SI-SDR` mide cuánta de la "distorsión" es coloración lineal inofensiva.

In [ ]:
from scipy.stats import wilcoxon

BASE = {'SI-SDR': 'base_SI-SDR_early', 'SDR': 'base_SDR_early'}

print('=' * 78)
print('1) ¿Se mueve el BASELINE con M?  (si se moviera, la Delta seria un artefacto)')
print('=' * 78)
print(df[df.processor == 'NM_MVDR']
        .groupby('M')[['base_SI-SDR_early', 'base_SDR_early', 'base_SIR_early']]
        .median().round(3).to_string())
print('\n-> plano en ~-0.82 dB. El microfono de referencia cambia de indice con M, pero')
print('   no de calidad: la Delta NO esta contaminada por el baseline.')

print('\n' + '=' * 78)
print('2) TEST PAREADO M=6 vs M=12 (misma escena, sala y topologia)')
print('=' * 78)
rows = []
for proc in PROCS:
    d = df[df.processor == proc]
    for m in ['SI-SDR', 'SDR', 'SIR', 'SAR']:
        P = d.pivot_table(index=['interf_scenario', 'rt60', 'topology'], columns='M',
                          values=f'Delta_tot_{m}_early').dropna()
        diff = P[M_LIST[-1]] - P[M_LIST[0]]
        rows.append({'procesador': proc, 'metrica': m,
                     'mediana M12-M6': float(np.median(diff)),
                     'IQR entre escenas': d[f'Delta_tot_{m}_early'].quantile(.75)
                                          - d[f'Delta_tot_{m}_early'].quantile(.25),
                     '% escenas que empeoran': 100 * float((diff < 0).mean()),
                     'p (Wilcoxon)': wilcoxon(diff).pvalue})
PAIR = pd.DataFrame(rows)
PAIR['|efecto| / IQR'] = PAIR['mediana M12-M6'].abs() / PAIR['IQR entre escenas']
show_table(PAIR.set_index(['procesador', 'metrica']), None, '{:.3f}')

print('\n-> El SI-SDR de NM-MVDR cae 0.09 dB y empeora en el 56 % de las escenas: a un pelo')
print('   del 50 % que daria tirar una moneda. Wilcoxon lo declara significativo solo')
print('   porque hay N = 720 pares. Comparar con el SIR, que sube 1.4 dB y empeora en el')
print('   19 % de las escenas: ESO es un efecto. El del SI-SDR es un empate.')
print('-> Y en el ORACLE el SI-SDR SUBE con M. Si la metrica estuviera rota, se romperia')
print('   tambien ahi. No se rompe: no hay nada malo con el SI-SDR.')

print('\n' + '=' * 78)
print('3) SDR vs SI-SDR: ¿cambia algo la conclusion si se reemplaza una por la otra?')
print('=' * 78)
d      = df[df.processor == 'NM_MVDR']
sisdr  = d['Delta_tot_SI-SDR_early']
sdr    = d['Delta_tot_SDR_early']
sir    = d['Delta_tot_SIR_early']
rho_ss = sisdr.corr(sdr,  method='spearman')
rho_is = sisdr.corr(sir,  method='spearman')
rho_ds = sdr.corr(sir,    method='spearman')
print(f'  Correlacion fila a fila SI-SDR vs SDR (Spearman): {rho_ss:+.3f}')
gap = (df.groupby(['processor', 'M'])
         .apply(lambda g: (g['proc_SDR_early'] - g['proc_SI-SDR_early']).median(),
                include_groups=False)
         .unstack('M'))
show_table(gap, 'Brecha SDR - SI-SDR [dB] = coloracion LINEAL que el filtro de 512 taps perdona',
           '{:.3f}')
show_table(d.groupby('topology', observed=True)[['Delta_tot_SI-SDR_early', 'Delta_tot_SDR_early']]
            .median().reindex(TOPOLOGIES),
           'Ranking de topologias segun cada una (NM-MVDR)', '{:+.3f}')

print('\n-> rho = 0.94: fila a fila son casi la misma medicion. Solo discrepan al agregar,')
print('   sobre un efecto de +/-0.1 dB que ya vimos que es ruido.')
print('-> La brecha SDR - SI-SDR crece con M en NM-MVDR (1.41 -> 1.62 dB) y casi no se')
print('   mueve en el oracle. Ahi si hay algo real, pero chico: con mas microfonos el')
print('   NM-MVDR agrega ~0.2 dB de coloracion LINEAL, que el SDR perdona y el SI-SDR no.')
print('   No es distorsion audible: es la respuesta del propio filtro espacial.')

print('\n' + '=' * 78)
print('CONCLUSION: se conserva el SI-SDR.')
print('=' * 78)
print('  - No esta roto: baseline estable, se comporta bien en el oracle, rho=0.94 con SDR.')
print('  - Es MENOS redundante que el SDR frente al SIR que ya se reporta')
print(f'    (rho SI-SDR/SIR = {rho_is:.2f} vs rho SDR/SIR = {rho_ds:.2f}): aporta mas informacion nueva.')
print('  - Para un BEAMFORMER, que el SDR perdone cualquier filtro de 512 taps es un')
print('    problema y no una virtud: perdona justamente la coloracion que introduce el')
print('    beamformer. El SI-SDR, que solo perdona una ganancia, es el criterio mas duro.')
print('  - El SDR queda como control cruzado en esta seccion, que es donde sirve.')

## 6. El eje crítico: elevación del target

Las cuatro topologías son **planares**, así que la elevación es su eje débil: la
proyección del desplazamiento angular sobre el plano del arreglo se achica con
`cos(elevación)`. Era acá, si en algún lado, donde el reparto de micrófonos tenía que
mostrar diferencias — y por eso el toroide se diseñó para barrer elevación de forma
controlada (≈ 5–43°) en lugar de sortear todo el hemisferio.

Un chequeo antes de leer las curvas: en el toroide la elevación **no** está confundida
con la distancia (`corr(elevación, distancia slant) ≈ −0,04`), así que lo que se ve es
efecto de ángulo, no de que las fuentes altas estén más cerca.

In [ ]:
fig, axes = plt.subplots(len(PROCS), 4, figsize=(14.0, 3.6 * len(PROCS)), squeeze=False)
for r, proc in enumerate(PROCS):
    sub = view[view.processor == proc]
    for c, metric in enumerate(METRICS):
        ax  = axes[r][c]
        piv = sub.pivot_table(index='el_mid', columns='topology', values=metric,
                              aggfunc='median', observed=True)
        for topo in TOPOLOGIES:
            if topo in piv.columns:
                ax.plot(piv.index, piv[topo].values, marker=TOPO_MARKER[topo],
                        color=TOPO_COLOR[topo], lw=1.9, ms=6.0, mec='white', mew=1.0,
                        label=TOPO_LABEL[topo])
        ax.set(xlabel='elevacion del target [deg]' if r == len(PROCS)-1 else '',
               ylabel=MSHORT[metric], title=metric if r == 0 else '')
        only_grid(ax)
top = head(fig, 'Dentro del rango realista (5–43°), la elevación no degrada a nadie',
           'Mediana por bin de 7,5° de elevación del target. PESQ y STOI son planos; SI-SDR y SIR '
           'incluso suben. El cono de ambigüedad del arreglo planar no llega a morder acá.')
head_legend(fig, axes[0][3], ncols=4)
fig.tight_layout(rect=(0.032, 0, 1, top))
row_labels(fig, axes, [PROC_LABEL[p] for p in PROCS], [PROC_INK[p] for p in PROCS])
plt.show()

# Cuanto cae cada metrica del bin mas bajo al mas alto (NM-MVDR)
sub  = view[view.processor == 'NM_MVDR']
piv  = sub.pivot_table(index='el_mid', columns='topology', values=METRICS,
                       aggfunc='median', observed=True)
caida = pd.DataFrame({m: piv[m].iloc[-1] - piv[m].iloc[0] for m in METRICS}).reindex(TOPOLOGIES)
show_table(caida, f'NM-MVDR — variacion del bin de elevacion mas BAJO al mas ALTO '
                  f'({piv.index[0]:.1f}° -> {piv.index[-1]:.1f}°; signo + = MEJORA al subir)',
           '{:+.3f}')

print('\nEl resultado esperado era una caida y no la hay: PESQ y STOI quedan planos y el')
print('SI-SDR MEJORA ~1 dB en las cuatro topologias. La lectura es que el toroide, por')
print('diseno, nunca acerca al target al cenit (se corta en ~43°), que es donde el cono de')
print('ambiguedad de un arreglo planar realmente duele. Dentro del rango de uso real la')
print('elevacion NO es un eje discriminante -- y lo poco que se mueve (el SIR de la')
print('circular sube +1.8 dB, el de la espiral no se mueve) va en la misma direccion que')
print('el resto del analisis: manda la apertura, no la elevacion.')

## 7. Robustez a la reverberación

Cada sala combina RT60 y volumen (oficina 6×7×2,8 m → salón 9×11×3,5 m). Al subir el
RT60, la parte coherente del campo se achica frente a la difusa: la matriz de covarianza
de ruido se vuelve más isótropa y el MVDR tiene menos estructura espacial que explotar.

Dos avisos para leer bien el gráfico:

1. **PESQ y SIR caen con el RT60, pero el SI-SDR sube.** No es una contradicción: estas
   son métricas **Δ**, y en la sala más reverberante el micrófono crudo — el punto de
   partida — es mucho peor. Hay más para recuperar, así que la *mejora* puede crecer
   aunque la calidad *absoluta* de la salida baje.
2. **RT60 y volumen no están separados** en este diseño: la sala de RT = 0,8 s también es
   la más grande. Lo que se ve es el efecto conjunto «sala difícil», no el de la
   absorción sola.

In [ ]:
fig, axes = plt.subplots(len(PROCS), 4, figsize=(13.6, 3.15 * len(PROCS)), squeeze=False)
for r, proc in enumerate(PROCS):
    sub = view[view.processor == proc]
    for c, metric in enumerate(METRICS):
        ax  = axes[r][c]
        piv = (sub.pivot_table(index='topology', columns='rt60', values=metric,
                               aggfunc='median', observed=True).reindex(TOPOLOGIES))
        im  = ax.imshow(piv.values, cmap=CMAP_SEQ, aspect='auto')
        lo, hi = np.nanmin(piv.values), np.nanmax(piv.values)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.values[i, j]
                ax.text(j, i, MFMT[metric].format(v), ha='center', va='center',
                        fontsize=8.6,
                        color='white' if (v - lo) / (hi - lo + 1e-12) > 0.58 else INK)
        ax.set_xticks(range(piv.shape[1]),
                      [f'{rt:.2f}\n{ROOM_LABEL[rt]}' for rt in piv.columns], fontsize=8.4)
        ax.set_yticks(range(piv.shape[0]), [TOPO_LABEL[t] for t in TOPOLOGIES], fontsize=8.6)
        for lbl, topo in zip(ax.get_yticklabels(), TOPOLOGIES):
            lbl.set_color(TOPO_COLOR[topo])
        if c: ax.set_yticklabels([])
        ax.set(title=metric if r == 0 else '',
               xlabel='RT60 [s]' if r == len(PROCS)-1 else '')
        ax.grid(False); ax.tick_params(length=0)
        for sp in ax.spines.values(): sp.set_visible(False)
top = head(fig, 'La sala mueve el resultado mucho más que la topología',
           'Mediana por (topología × sala). Dentro de cada columna los cuatro valores casi no se '
           'mueven; entre columnas, sí. Escala de color independiente por panel.')
fig.tight_layout(rect=(0.055, 0, 1, top))
row_labels(fig, axes, [PROC_LABEL[p] for p in PROCS], [PROC_INK[p] for p in PROCS])
plt.show()

sub = view[view.processor == 'NM_MVDR']
show_table(sub.pivot_table(index='rt60', columns='topology', values=METRICS,
                           aggfunc='median', observed=True)
              .reindex(columns=pd.MultiIndex.from_product([METRICS, TOPOLOGIES])),
           'NM-MVDR — mediana por (RT60 x topologia)', '{:+.3f}')

## 8. Dispersión escena a escena

La mediana **rankea**, pero no dice nada sobre **consistencia**. Una topología que gana
por poco en mediana y falla feo en el 10% peor de las escenas es peor producto que una
que empata y nunca se cae. Estas cajas muestran el reparto completo de las 540 escenas
por topología: caja = cuartiles, línea = mediana, bigotes = percentiles 5 y 95.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14.0, 4.6))
off = {p: (i - (len(PROCS) - 1) / 2) * 0.34 for i, p in enumerate(PROCS)}
for ax, metric in zip(axes, METRICS):
    for proc in PROCS:
        sub = view[view.processor == proc]
        data = [sub[sub.topology == t][metric].dropna().values for t in TOPOLOGIES]
        pos  = np.arange(len(TOPOLOGIES)) + off[proc]
        oracle = proc != 'NM_MVDR'
        bp = ax.boxplot(data, positions=pos, widths=0.29, patch_artist=True,
                        whis=(5, 95), showfliers=False,
                        medianprops=dict(color='white' if not oracle else INK, lw=1.5),
                        whiskerprops=dict(color=MUTED, lw=1.0),
                        capprops=dict(color=MUTED, lw=1.0))
        for patch, topo in zip(bp['boxes'], TOPOLOGIES):
            patch.set_facecolor(PROC_COLOR[proc] if oracle else TOPO_COLOR[topo])
            patch.set_alpha(0.95 if not oracle else 0.65)
            patch.set_edgecolor('white'); patch.set_linewidth(0.9)
    ax.set_xticks(range(len(TOPOLOGIES)), [TOPO_LABEL[t] for t in TOPOLOGIES], rotation=18,
                  ha='right')
    for lbl, topo in zip(ax.get_xticklabels(), TOPOLOGIES):
        lbl.set_color(TOPO_COLOR[topo])
    ax.set(ylabel=MSHORT[metric], title=metric)
    only_grid(ax)   # sin linea en 0: todos los valores son positivos y comprimiria la escala
axes[0].plot([], [], 's', color=TOPO_COLOR['circular'], label='NM-MVDR (caja en color)')
axes[0].plot([], [], 's', color=PROC_COLOR['SOUDEN_ORACLE_SCM'], label='Souden oracle (caja gris)')
top = head(fig, 'La escena manda: dentro de una topología, el reparto es enorme',
           'Cajas = cuartiles, bigotes = P5–P95 sobre 540 escenas. El ancho de cada caja '
           'empequeñece cualquier diferencia entre las medianas de las cuatro topologías, y la '
           'distancia al techo oracle es mayor que todas ellas juntas.')
head_legend(fig, axes[0], ncols=2)
fig.tight_layout(rect=(0, 0, 1, top)); plt.show()

disp = []
for proc in PROCS:
    sub = view[view.processor == proc]
    for metric in METRICS:
        entre_topo = sub.groupby('topology', observed=True)[metric].median()
        disp.append({'procesador': proc, 'metrica': metric,
                     'rango entre topologias': entre_topo.max() - entre_topo.min(),
                     'IQR entre escenas': sub[metric].quantile(.75) - sub[metric].quantile(.25),
                     'P5-P95 entre escenas': sub[metric].quantile(.95) - sub[metric].quantile(.05)})
DISP = pd.DataFrame(disp)
DISP['topologia / IQR  [%]'] = 100 * DISP['rango entre topologias'] / DISP['IQR entre escenas']
show_table(DISP.set_index(['procesador', 'metrica']),
           'Cuanto pesa la topologia frente a la variabilidad de la escena', '{:.3f}')

## 9. Síntesis

La tabla consolida el ranking (puesto 1 = mejor por métrica) y lo pone al lado de la
magnitud real de las diferencias, para que el número no se lea fuera de escala.

In [ ]:
rank = med(view[view.processor == 'NM_MVDR'], 'topology').reindex(TOPOLOGIES)
rk   = rank.rank(ascending=False).astype(int)
rk.columns = [f'#{c}' for c in rk.columns]
RES  = pd.concat([rank, rk], axis=1)
RES['puntaje'] = rk.sum(axis=1)
# Desempate explicito: a igual puntaje, gana el mejor SIR (la metrica con la
# diferencia entre topologias mas grande respecto de su propia dispersion).
RES = RES.sort_values(['puntaje', 'SIR'], ascending=[True, False])
show_table(RES, 'NM-MVDR — ranking consolidado (menor puntaje = mejor; desempate por SIR)',
           '{:+.3f}')

sub      = view[view.processor == 'NM_MVDR']
ganador  = RES.index[0]
empatan  = list(RES.index[RES['puntaje'] == RES['puntaje'].min()])
peor     = RES.index[-1]
rel      = {m: 100 * (sub.groupby('topology', observed=True)[m].median().max()
                      - sub.groupby('topology', observed=True)[m].median().min())
               / (sub[m].quantile(.75) - sub[m].quantile(.25)) for m in METRICS}

print('\n' + '=' * 78)
print('SINTESIS')
print('=' * 78)
print(f'1. RANKING. Empatan arriba {" y ".join(empatan)} '
      f'({RES["puntaje"].min()}/{4*len(METRICS)} puntos); desempata la grilla por SIR.')
print(f'   La {peor} es la unica que pierde de forma consistente: ultima en las 4 metricas.')
print()
print('2. MAGNITUD. Mejor menos peor topologia, en % del IQR entre escenas:')
for m in METRICS:
    marca = 'relevante' if rel[m] > 40 else ('marginal' if rel[m] > 20 else 'despreciable')
    print(f'   {m:>7}: {rel[m]:5.1f} %  -> {marca}')
print('   Solo el SIR discrimina de verdad. En PESQ, STOI y SI-SDR las cuatro topologias')
print('   son intercambiables a efectos practicos.')
print('   (El SI-SDR se verifico contra el SDR en la seccion 5.1: no esta subestimando')
print('    nada, las dos metricas correlacionan a rho = 0.94 fila a fila.)')
print()
print('3. POR QUE. El ranking de SIR sigue a la APERTURA MAXIMA (Spearman rho = +0.79')
print('   global, +0.95 a M fijo), no a la densidad de muestreo espacial. Con el area')
print('   fija lo que rinde es ESTIRAR el arreglo, no amontonar micros: la grilla gana')
print('   porque al igualarse por AREA (y no por inscripcion) llega a 20-24 cm de')
print('   apertura; la espiral pierde porque concentra micros cerca del centro y se')
print('   queda en ~13 cm.')
print()
print('4. EL ORDEN DE LOS FACTORES. De mayor a menor efecto sobre el resultado:')
_t = sub.groupby('topology', observed=True)['SIR'].median()
_m = sub.groupby('M', observed=True)['SIR'].median()
print('   la escena (IQR ~ 3.3 dB de SIR) > la sala / RT60 (~3 dB de 0.3 a 0.8 s) >')
print(f'   la topologia ({_t.max()-_t.min():.1f} dB entre la mejor y la peor) ~ el numero de')
print(f'   micros M ({_m[M_LIST[-1]]-_m[M_LIST[0]]:.1f} dB al DUPLICARLO, de {M_LIST[0]} a {M_LIST[-1]}).')
print('   Topologia y M pesan PARECIDO, y los dos pesan poco. Pero la topologia es')
print('   gratis y M cuesta canales de ADC y MACs: a igualdad de efecto, conviene')
print(f'   gastar el presupuesto en la geometria. Ojo: {_t.max()-_t.min():.1f} dB incluye a la espiral;')
print(f'   entre las dos mejores (grilla y circular) la diferencia es de solo {_t["grid"]-_t["circular"]:.2f} dB.')
print()
print('5. EL ORACLE RANKEA IGUAL. Con mascara ideal el orden de topologias se repite y')
print('   las diferencias siguen siendo del mismo tamano. El empate no lo causa la')
print('   estimacion de mascara del DTLN: es geometrico. Ninguna topologia planar de')
print(f'   {A_REF*1e4:.0f} cm2 tiene mucho mas que ofrecer que otra.')
print()
print('6. RECOMENDACION. Grilla rectangular. Gana o empata en las 4 metricas, es la mas')
print('   simple de fabricar y rutear en PCB, y su ventaja tiene una causa entendida')
print('   (apertura) en vez de ser ruido de muestreo. Con M = 8 ya se captura casi todo:')
print(f'   pasar a M = 12 agrega {sub[sub.topology=="grid"].groupby("M", observed=True)["SIR"].median()[12] - sub[sub.topology=="grid"].groupby("M", observed=True)["SIR"].median()[8]:+.2f} dB de SIR por 4 canales mas de hardware.')

---

### Qué queda fuera de este notebook

- **DS (delay-and-sum)** no está en el dataset: el barrido corrió solo `NM_MVDR` y
  `SOUDEN_ORACLE_SCM`. Si hiciera falta el piso de referencia hay que volver a correr
  el notebook de simulación con `DS()` en `processors`.
- **Post-filtro DTLN** desactivado (`apply_dtln_post=False`): acá se mide el
  beamformer solo, sin el post-procesado ([[stft-window-coupling-dtln]]).
- **SI-SDR y SIR se miden sobre un único canal de referencia** (el centroide). Es la
  convención de todo el proyecto ([[metrics-ref-mic-mismatch]]) y es lo que hace
  comparables estas cifras con las de las Fases 1–3.